# 第 19 章　前処理と交差検証

前章の組合せモデルは，貸出件数を 1 時間当たり平均約 92 件外しました。貸出が多い時期にも同じくらいの誤差で予測できるでしょうか。講義では，前章の符号化と回帰を一つの手順にまとめ，学習する日を変えて，その後の 3 か月ずつを検証します。実習では，二つの期間と一つの時間帯を選び，その時刻に絞ると，二つの期間でモデルの外れ方はどう違うかを調べます。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。ローカルでは配布 ZIP の配置で開くか，このノートと CSV を同じフォルダーに置きます。手元にファイルがなければ公開 URL から読み込みます。

## 講義　同じ手順を違う時期に使う

### 貸出の表と組合せ

前日の終了時に，翌日の各時間帯の貸出件数 `cnt` を予測する設定です。モデルの入力にするのは，前日のうちに分かる時刻と日種別の組合せです。データから 2011 年 1 月〜2012 年 6 月の収録行を読みます。行のない時間帯には 0 件を補いません。春には 2 年分を調べましたが，2012 年後半は秋のモデル比較に使わずに取っておきます。

前半では 2011 年で学習し，2012 年 1〜6 月を検証します。`hour_day` の先頭を見て，どの時刻・日種別を表すかを確かめます。

In [ ]:
from pathlib import Path

import pandas as pd

file_name = "bike-sharing-hourly.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

hourly = pd.read_csv(
    csv_path,
    usecols=["dteday", "hr", "workingday", "cnt"],
    parse_dates=["dteday"],
)
hourly = hourly.loc[hourly["dteday"] < "2012-07-01"].copy()
hourly.isna().sum()

In [ ]:
hourly["hour_day"] = (
    hourly["hr"].astype(str).str.zfill(2)
    + "_"
    + hourly["workingday"].astype(str)
)
features = ["hour_day"]
train = hourly.loc[hourly["dteday"] < "2012-01-01"]
valid = hourly.loc[hourly["dteday"] >= "2012-01-01"]

X_train = train[features]
y_train = train["cnt"]
X_valid = valid[features]
y_valid = valid["cnt"]
X_train.head(3)

### 符号化と回帰をつなぐ

はじめに，符号化器と回帰モデルを別々に呼びます。学習データで目印の作り方を決めてから，同じ規則で学習と検証の両方を変換します。次に `Pipeline` で，同じ二つの処理をつなぎます。`fit()` と `predict()` のうち，符号化の規則を決めるのはどちらですか。最後のセルでは，二通りの手順で同じ予測値と MAE になるか確かめます。

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(drop="first", sparse_output=False).set_output(transform="pandas")
encoded_train = encoder.fit_transform(X_train)
encoded_valid = encoder.transform(X_valid)

regression = LinearRegression()
regression.fit(encoded_train, y_train)
separate_prediction = regression.predict(encoded_valid)

print("Training:", encoded_train.shape)
print("Validation:", encoded_valid.shape)

In [ ]:
from sklearn.pipeline import Pipeline

def make_model():
    encoder = OneHotEncoder(drop="first", sparse_output=False).set_output(transform="pandas")
    return Pipeline([
        ("encode", encoder),
        ("regression", LinearRegression()),
    ])

model = make_model()
model.fit(X_train, y_train)
fixed_prediction = model.predict(X_valid)

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error

print("Same predictions:", np.allclose(separate_prediction, fixed_prediction))
print(f"Separate steps MAE: {mean_absolute_error(y_valid, separate_prediction):.2f}")
print(f"Pipeline MAE:       {mean_absolute_error(y_valid, fixed_prediction):.2f}")

example = valid[["dteday", "hour_day", "cnt"]].head(3).copy()
example["separate_prediction"] = separate_prediction[:3]
example["pipeline_prediction"] = fixed_prediction[:3]
example

### 補完値を学習側から決める

貸出データの入力には空欄がありません。ここでは別の架空例で，特徴量に欠損があるとき，学習に渡す表によって補完値がどう変わるかを確かめます。学習側の二つの値 2，4 の平均と，後から届く検証側の 100 を加えた平均を手で求めます。二つのうちどちらなら，学習した時点で決められますか。コードを実行し，`fit()` に渡す表と，`transform()` の出力を見比べます。

In [ ]:
from sklearn.impute import SimpleImputer

toy_train = pd.DataFrame({"x": [2.0, 4.0, float("nan")]})
toy_valid = pd.DataFrame({"x": [100.0, float("nan")]})

imputer = SimpleImputer(strategy="mean")
imputer.fit(toy_train)
print("Value learned:", imputer.statistics_[0])
print("Training:", imputer.transform(toy_train)[:, 0])
print("Validation:", imputer.transform(toy_valid)[:, 0])

### 日付で分けて学び直す

四つの検証期間を日付で指定します。各期間の開始直前までの記録で学び，その後の 3 か月で予測を試します。たとえば 2011 年 8 月の記録は，図の一番上の分割ではどちらに入り，二番目の分割ではどちらに入りますか。

`evaluate_period()` は，一組分の学習と検証を実行する関数です。呼び出すたびに，新しく作った組合せモデルをその期間で学習します。一律平均も同じ学習側から計算します。全時刻を調べる講義では，検証の時刻を指定する引数 `evaluation_hour` を省略します。

In [ ]:
periods = [
    ("2011 Jul–Sep", "2011-07-01", "2011-10-01"),
    ("2011 Oct–Dec", "2011-10-01", "2012-01-01"),
    ("2012 Jan–Mar", "2012-01-01", "2012-04-01"),
    ("2012 Apr–Jun", "2012-04-01", "2012-07-01"),
]

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7.2, 3.2)

first_day = pd.Timestamp("2011-01-01")
fig, ax = plt.subplots(layout="constrained")
for number, (_, start, stop) in enumerate(periods, start=1):
    start = pd.Timestamp(start)
    stop = pd.Timestamp(stop)
    ax.barh(number, (start - first_day).days, left=first_day,
            height=0.55, color="#286b70", label="Training" if number == 1 else None)
    ax.barh(number, (stop - start).days, left=start,
            height=0.55, color="#f4dbca", edgecolor="#a34f30", hatch="//",
            label="Validation" if number == 1 else None)
ax.set(xlabel="Date", ylabel="Split", yticks=[1, 2, 3, 4],
       xlim=(first_day, pd.Timestamp("2012-07-05")))
ax.set_xticks(pd.date_range("2011-01-01", "2012-07-01", freq="QS"))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.set_axisbelow(True)
ax.grid(axis="x", color="#d5dce2")
ax.invert_yaxis()
ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1), ncols=2)
plt.show()

In [ ]:
def evaluate_period(data, valid_start, valid_stop, evaluation_hour=None):
    training = data.loc[data["dteday"] < valid_start]
    validation = data.loc[
        (data["dteday"] >= valid_start) & (data["dteday"] < valid_stop)
    ]
    if evaluation_hour is not None:
        validation = validation.loc[validation["hr"] == evaluation_hour]

    period_model = make_model()
    period_model.fit(training[features], training["cnt"])
    prediction = period_model.predict(validation[features])
    baseline = training["cnt"].mean()

    observed = validation["cnt"]
    residual = observed - prediction
    return {
        "train_rows": len(training),
        "valid_rows": len(validation),
        "baseline_mae": mean_absolute_error(observed, [baseline] * len(observed)),
        "model_mae": residual.abs().mean(),
        "mean_residual": residual.mean(),
        "mean_observed": observed.mean(),
        "mean_prediction": prediction.mean(),
    }

In [ ]:
results = []
for label, start, stop in periods:
    score = evaluate_period(hourly, start, stop)
    results.append({"period": label, **score})

comparison = pd.DataFrame(results).set_index("period")
mae_columns = ["baseline_mae", "model_mae"]
comparison[["train_rows", "valid_rows"] + mae_columns].round(2)

### 二つの方法の誤差を期間ごとに読む

縦の線分は，同じ期間の二つの MAE を結んでいます。線分の長さで一律平均との差を，丸の高さで組合せモデルの誤差を読みます。四期間で，基準との差の大きさと，組合せモデル自体の誤差の大きさは，それぞれどう動きましたか。四つの MAE を一つにまとめるとき，期間に同じ重さを付ける方法と，各行に同じ重さを付ける方法も比べます。

In [ ]:
plt.rcParams["figure.figsize"] = (7.2, 3.8)

period_positions = range(len(comparison))
period_labels = [label.replace(" ", "\n", 1) for label in comparison.index]
fig, ax = plt.subplots(layout="constrained")
ax.vlines(period_positions, comparison["model_mae"], comparison["baseline_mae"],
          color="#87939d", linewidth=1.5, zorder=2)
ax.scatter(period_positions, comparison["baseline_mae"], marker="s", s=46,
           color="#687581", label="Constant baseline", zorder=3)
ax.scatter(period_positions, comparison["model_mae"], marker="o", s=51,
           color="#a34f30", label="Hour × day type", zorder=3)
ax.set(xlabel="Validation period", ylabel="MAE (rentals per hour)",
       xlim=(-0.4, 3.4), ylim=(0, 210), xticks=list(period_positions),
       xticklabels=period_labels)
ax.grid(axis="y", color="#dde4e8")
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper left")
plt.show()

In [ ]:
mean_by_period = comparison["model_mae"].mean()
total_absolute_error = (comparison["model_mae"] * comparison["valid_rows"]).sum()
mean_by_row = total_absolute_error / comparison["valid_rows"].sum()

print(f"One vote per period: {mean_by_period:.2f}")
print(f"One vote per row:    {mean_by_row:.2f}")

### 足りない方に外れた期間を調べる

実測の平均と組合せモデルの予測の平均を比べます。予測が少なかったことが平均残差の正負のどちらで表れるかを確認します。2012 年 4〜6 月の誤差の大きさと残差の符号は，準備をする人にとってどんな違いになりますか。

In [ ]:
comparison[["mean_observed", "mean_prediction", "mean_residual"]].round(2)

In [ ]:
plt.rcParams["figure.figsize"] = (7.2, 3.6)

fig, ax = plt.subplots(layout="constrained")
ax.scatter(period_positions, comparison["mean_observed"], marker="o", s=50,
           facecolors="white", edgecolors="#286b70", linewidths=1.5, zorder=3,
           label="Observed mean")
ax.scatter(period_positions, comparison["mean_prediction"], marker="+", s=65,
           color="#a34f30", linewidths=1.7, zorder=4, label="Predicted mean")
ax.set(xlabel="Validation period", ylabel="Mean rentals per hour",
       xlim=(-0.4, 3.4), ylim=(0, 295), xticks=list(period_positions),
       xticklabels=period_labels)
ax.grid(axis="y", color="#dde4e8")
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper left")
plt.show()

学習に使う行を増やした結果を読むには，どの行で評価するかをそろえます。前半で使った 2011 年末のモデルと，2012 年 3 月末で学習し直したモデルから，それぞれ 2012 年 4〜6 月の予測を取り出して比べます。

In [ ]:
apr_to_jun = valid["dteday"] >= "2012-04-01"
mae_without_update = mean_absolute_error(
    valid.loc[apr_to_jun, "cnt"], fixed_prediction[apr_to_jun]
)
mae_with_update = comparison.loc["2012 Apr–Jun", "model_mae"]
print(f"Same {apr_to_jun.sum():,} validation rows")
print(f"Trained through Dec 2011: {mae_without_update:.2f}")
print(f"Trained through Mar 2012: {mae_with_update:.2f}")
print(f"Reduction: {mae_without_update - mae_with_update:.2f}")

## 実習　二期間で同じ時間帯を比べる

### 1. 期間と時間帯を選ぶ（約 5 分）

次の四つから，知りたい二つの検証期間を選びます。どちらの期間でも調べたい時間帯を一つ選びます。たとえば，朝に貸出に備えるなら 8 時台，昼の利用が気になるなら 13 時台を選べます。

1. 2011 年 7〜9 月
2. 2011 年 10〜12 月
3. 2012 年 1〜3 月
4. 2012 年 4〜6 月

講義で見た全時刻の結果を手掛かりに，選んだ時間帯でも同じような違いが出そうか予想します。期間と時間帯を選んだ理由も書きます。

（選んだ二期間と時間帯，理由と予想を記入）

### 2. 二つの期間を検証する（約 9 分）

次のセルの三つの `None` を，期間番号二つと時刻（0〜23）に替えます。選んだ時間帯は，どちらの期間にも同じ値を渡します。期日と時刻を指定して `evaluate_period()` を呼ぶ行を読み，どこまでの記録で `fit()` するか，指定した時刻の行だけになるのは学習と検証のどちらかを説明します。

表の `train_rows` は学習行数，`valid_rows` は選んだ時刻の検証行数です。各期間の二つの MAE と，組合せモデルの平均残差 `mean_residual` を読みます。

In [ ]:
first_period = None
second_period = None
target_hour = None

if not all(isinstance(value, int) for value in [first_period, second_period, target_hour]):
    print("期間番号を二つ選び，0〜23 の時刻を指定します。")
elif first_period not in range(1, 5) or second_period not in range(1, 5):
    print("期間番号は 1〜4 から指定します。")
elif first_period == second_period or not 0 <= target_hour <= 23:
    print("別の二期間と，0〜23 の時刻を指定します。")
else:
    chosen_scores = []
    for number in [first_period, second_period]:
        label, start, stop = periods[number - 1]
        score = evaluate_period(hourly, start, stop, evaluation_hour=target_hour)
        chosen_scores.append({"period": label, **score})

    chosen_comparison = pd.DataFrame(chosen_scores).set_index("period")
    output_columns = ["train_rows", "valid_rows", "baseline_mae", "model_mae", "mean_residual"]
    print(f"Validation at {target_hour:02d}:00")
    display(chosen_comparison[output_columns].round(2))

### 3. 担当者へ結果を伝える（約 6 分）

選んだ時間帯の準備をする人を想定し，3〜4 文で報告します。

1. 選んだ時間帯と二つの検証期間を示し，それぞれで何行を調べたか，基準と組合せモデルの MAE はどのくらいだったかを書く。
2. 講義の全時刻で見えた違いや自分の予想と比べて，どんな違いが残ったかを説明する。平均残差も使い，予測が多め・少なめのどちらへ外れたかを述べる。
3. 準備に使うために，次はどの期間の記録で，何をそろえて比べたいかを書く。

（ここに報告を記入）

カーネルを再起動し，上から全セルを実行します。選んだ時間帯の表と報告を残したノートブックを提出します。

### データの出典

Capital Bikeshare の時間別集計は，Hadi Fanaee-T が UCI Machine Learning Repository で公開した `hour.csv` を教材用に改名したものです。 [データ配布元](https://doi.org/10.24432/C5W894) ・ [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)